In [34]:
from pathlib import Path
import pickle
from scipy import stats
from microtool.monte_carlo.IO import make_expirement_directories
from microtool.monte_carlo.simulation import MonteCarloSimulation

# 1. Resolve experiment directories
plotdir, modeldir, simdir, schemedir = make_expirement_directories(
    Path.cwd() / "VERDICT", "Test_Initial"
)

# 2. Load model and schemes from disk
with open(modeldir / "model.pkl", "rb") as f:
    model = pickle.load(f)

with open(schemedir / "initial_scheme.pkl", "rb") as f:
    initial_scheme = pickle.load(f)

with open(schemedir / "optimal_scheme.pkl", "rb") as f:
    optimal_scheme = pickle.load(f)

# 3. Update model ground truth and define noise distribution
model["radius"].value = 10
noise_distribution = stats.norm(0, scale=0.02)

# 4. Run Monte Carlo simulation for the initial scheme
simulation = MonteCarloSimulation(
    initial_scheme,
    model,
    noise_distribution,
    n_sim=1000,
    fit_options={"diff_step": 1e-4},
)
simulation.run()
initial = simulation.result
simulation.save(simdir / "initial.pkl")

# 5. Run Monte Carlo simulation for the optimal scheme
simulation.set_scheme(optimal_scheme)
simulation.run()
optimal = simulation.result
simulation.save(simdir / "optimal.pkl")

# 6. Re-load the saved results directly from disk
with open(simdir / "initial.pkl", "rb") as f:
    initial_sim = pickle.load(f)

with open(simdir / "optimal.pkl", "rb") as f:
    optimal_sim = pickle.load(f)

initial = initial_sim.result if hasattr(initial_sim, "result") else initial_sim
optimal = optimal_sim.result if hasattr(optimal_sim, "result") else optimal_sim

Starting Monte Carlo with 1000 simulations: 100%|██████████| 1000/1000 [00:08<00:00, 118.75it/s]


In [ ]:
from pathlib import Path
import pickle
import numpy as np
from matplotlib import pyplot as plt
from microtool.monte_carlo.IO import make_expirement_directories

# 1. Resolve experiment directories
plotdir, modeldir, simdir, schemedir = make_expirement_directories(
    Path.cwd() / "VERDICT", "Test_SOMA_t2"
)

# 2. Load model
with open(modeldir / "model.pkl", "rb") as f:
    model = pickle.load(f)

# 3. Load simulation results directly via pickle
with open(simdir / "initial.pkl", "rb") as f:
    initial_sim_data = pickle.load(f)

with open(simdir / "optimal.pkl", "rb") as f:
    optimal_sim_data = pickle.load(f)

# Handle cases where the pickle contains either the MonteCarloSimulation object or the DataFrame directly
initial_result = initial_sim_data.result if hasattr(initial_sim_data, "result") else initial_sim_data
optimal_result = optimal_sim_data.result if hasattr(optimal_sim_data, "result") else optimal_sim_data

# 4. Extract parameters and derive f_2
radius_key = "radius"

initial_f2 = 1 - initial_result["f_0"] - initial_result["f_1"]
optimal_f2 = 1 - optimal_result["f_0"] - optimal_result["f_1"]

radius_ref = model[radius_key].value if radius_key in model else model["model_radius"].value
f0_ref = model["model_f_0"].value
f1_ref = model["model_f_1"].value
f2_ref = 1 - f0_ref - f1_ref

plots_data = [
    (
        "radius",
        radius_ref,
        initial_result[radius_key],
        optimal_result[radius_key],
        np.linspace(0.2 * radius_ref, 2 * radius_ref, 50),
    ),
    (
        "f_0",
        f0_ref,
        initial_result["f_0"],
        optimal_result["f_0"],
        np.linspace(0, 1, 50),
    ),
    (
        "f_1",
        f1_ref,
        initial_result["f_1"],
        optimal_result["f_1"],
        np.linspace(0, 1, 50),
    ),
    (
        "f_2",
        f2_ref,
        initial_f2,
        optimal_f2,
        np.linspace(0, 1, 50),
    ),
]

# 5. Plot 2x2 comparison grid
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, (name, ref, init_vals, opt_vals, bins) in zip(axes, plots_data):
    init_rmse = np.sqrt(np.mean((init_vals - ref) ** 2))
    opt_rmse = np.sqrt(np.mean((opt_vals - ref) ** 2))

    ax.axvline(ref, color="black", linestyle="--", label="Reference")
    ax.hist(
        init_vals,
        bins=bins,
        alpha=0.5,
        label=f"Initial (RMSE={init_rmse:.3f})",
    )
    ax.hist(
        opt_vals,
        bins=bins,
        alpha=0.5,
        label=f"Optimal (RMSE={opt_rmse:.3f})",
    )

    ax.set_title(name)
    ax.set_xlabel(name)
    ax.set_ylabel("Count")
    ax.legend()

plt.tight_layout()
plt.savefig(plotdir / "parameter_histograms.png", dpi=300)
plt.show()

AttributeError: module 'numpy._globals' has no attribute '_signature_descriptor'

ImportError: numpy._core.multiarray failed to import

In [1]:
from pathlib import Path
import pickle
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import numpy.core._multiarray_umath as umath

from importlib.util import find_spec

for name in ("numpy.core.multiarray", "numpy._core.multiarray"):
    spec = find_spec(name)
    print(name, spec.origin if spec else "not found")


def extract_scheme_parameters(scheme):
    """
    Extracts b-values, echo times, effective diffusion times (Delta - delta/3),
    and gradient vectors directly from DiffusionAcquisitionScheme properties.
    Ensures timings are in milliseconds and b-values in s/mm².
    """
    # 1. b-values (property docstring: s/mm²)
    b_vals = np.asarray(scheme.b_values, dtype=float).flatten()
    if np.nanmax(b_vals) > 50000:
        b_vals = b_vals / 1e6

    # 2. Echo times (seconds -> ms)
    te_vals = np.asarray(scheme.echo_times, dtype=float).flatten()
    if np.nanmax(te_vals) < 1.0 and np.any(te_vals > 0):
        te_vals = te_vals * 1e3

    # 3. Pulse interval (Delta) and pulse duration (delta) (seconds -> ms)
    big_delta = np.asarray(scheme.pulse_intervals, dtype=float).flatten()
    small_delta = np.asarray(scheme.pulse_durations, dtype=float).flatten()

    if np.nanmax(big_delta) < 1.0 and np.any(big_delta > 0):
        big_delta = big_delta * 1e3

    if np.nanmax(small_delta) < 1.0 and np.any(small_delta > 0):
        small_delta = small_delta * 1e3

    # Effective diffusion time: Delta - delta / 3 (ms)
    delta_eff = big_delta - (small_delta / 3.0)

    # 4. Directions and unit vectors
    b_vecs = np.asarray(scheme.b_vectors, dtype=float)
    norms = np.linalg.norm(b_vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    unit_dirs = b_vecs / norms

    return b_vals, te_vals, delta_eff, b_vecs, unit_dirs


def aggregate_envelope_points(
    b, te, delta, b_tol=10.0, te_tol=0.5, delta_tol=0.2, base_size=70, scale_factor=35
):
    """Groups points with tight tolerances so distinct shells are preserved."""
    b_bins = np.round(b / b_tol) * b_tol
    te_bins = np.round(te / te_tol) * te_tol
    delta_bins = np.round(delta / delta_tol) * delta_tol

    coords = np.column_stack([b_bins, te_bins, delta_bins])
    unique_coords, inverse_indices, counts = np.unique(
        coords, axis=0, return_inverse=True, return_counts=True
    )

    n_clusters = len(unique_coords)
    agg_b = np.zeros(n_clusters)
    agg_te = np.zeros(n_clusters)
    agg_delta = np.zeros(n_clusters)

    for i in range(n_clusters):
        mask = inverse_indices == i
        agg_b[i] = np.mean(b[mask])
        agg_te[i] = np.mean(te[mask])
        agg_delta[i] = np.mean(delta[mask])

    agg_sizes = base_size + (counts - 1) * scale_factor

    return agg_b, agg_te, agg_delta, agg_sizes, counts


def plot_concentric_envelope_scatter(
    ax, agg_b, agg_te, agg_delta, agg_sizes, counts, norm, is_opt=False
):
    """
    Renders coincident shells at identical (b, TE) locations as concentric bullseye markers.
    Single shells are plotted as standard circles.
    """
    # Group shells that share the same (b, TE) space within tolerance
    b_grid = np.round(agg_b / 15.0) * 15.0
    te_grid = np.round(agg_te / 1.0) * 1.0
    coords_2d = np.column_stack([b_grid, te_grid])

    _, group_indices = np.unique(coords_2d, axis=0, return_inverse=True)
    n_groups = len(np.unique(group_indices))

    last_sc = None

    for g in range(n_groups):
        idx = np.where(group_indices == g)[0]

        if len(idx) == 1:
            # Single shell: standard marker
            i = idx[0]
            x, y = agg_b[i], agg_te[i]
            d_val = agg_delta[i]
            cnt = counts[i]

            last_sc = ax.scatter(
                x,
                y,
                c=[d_val],
                cmap="viridis",
                norm=norm,
                s=agg_sizes[i],
                edgecolors="black",
                linewidths=1.2,
                alpha=0.9,
                zorder=3,
            )

            label_col = "#b00000" if (is_opt and x < 10.0) else "#222222"
            ax.annotate(
                f"{cnt}x\n({d_val:.1f}ms)",
                xy=(x, y),
                xytext=(10, 6),
                textcoords="offset points",
                fontsize=8.5,
                fontweight="bold",
                color=label_col,
                zorder=4,
            )

        else:
            # Overlapping shells: sort so the one with more directions/larger size is outer
            sorted_sub = idx[np.argsort(-agg_sizes[idx])]
            outer_i = sorted_sub[0]
            inner_i = sorted_sub[1]

            x, y = agg_b[outer_i], agg_te[outer_i]

            # 1. Outer Ring (large disk)
            outer_size = max(agg_sizes[outer_i], 220)
            last_sc = ax.scatter(
                x,
                y,
                c=[agg_delta[outer_i]],
                cmap="viridis",
                norm=norm,
                s=outer_size,
                edgecolors="black",
                linewidths=1.5,
                alpha=0.9,
                zorder=3,
            )

            # 2. Inner Core (smaller concentric disk)
            inner_size = outer_size * 0.38
            ax.scatter(
                x,
                y,
                c=[agg_delta[inner_i]],
                cmap="viridis",
                norm=norm,
                s=inner_size,
                edgecolors="black",
                linewidths=1.2,
                alpha=0.95,
                zorder=4,
            )

            # 3. Clean unified annotation showing both shells
            label_text = (
                f"Outer: {counts[outer_i]}x ({agg_delta[outer_i]:.1f}ms)\n"
                f"Inner: {counts[inner_i]}x ({agg_delta[inner_i]:.1f}ms)"
            )

            ax.annotate(
                label_text,
                xy=(x, y),
                xytext=(14, 10),
                textcoords="offset points",
                fontsize=8,
                fontweight="bold",
                color="#111111",
                bbox=dict(
                    boxstyle="round,pad=0.25",
                    fc="white",
                    ec="gray",
                    lw=0.8,
                    alpha=0.85,
                ),
                arrowprops=dict(
                    arrowstyle="->",
                    color="black",
                    lw=0.8,
                    shrinkA=4,
                    shrinkB=4,
                ),
                zorder=5,
            )

    return last_sc


# -------------------------------------------------------------
# 1. Resolve Directories & Load Schemes
# -------------------------------------------------------------
plotdir, modeldir, simdir, schemedir = make_expirement_directories(
    Path.cwd() / "VERDICT", "Test_SOMA_t2_check"
)

with open(schemedir / "initial_scheme.pkl", "rb") as f:
    initial_scheme = pickle.load(f)

with open(schemedir / "optimal_scheme.pkl", "rb") as f:
    optimal_scheme_SOMA = pickle.load(f)

b_init, te_init, d_init, _, udirs_init = extract_scheme_parameters(initial_scheme)
b_opt, te_opt, d_opt, _, udirs_opt = extract_scheme_parameters(optimal_scheme_SOMA)

# -------------------------------------------------------------
# 2. Filter Both Schemes Using Baseline Mask
# -------------------------------------------------------------
keep_mask = b_init > 1e-2

b_init_f = b_init[keep_mask]
te_init_f = te_init[keep_mask]
d_init_f = d_init[keep_mask]
udirs_init_f = udirs_init[keep_mask]

b_opt_f = b_opt[keep_mask]
te_opt_f = te_opt[keep_mask]
d_opt_f = d_opt[keep_mask]
udirs_opt_f = udirs_opt[keep_mask]

# Aggregate points with tight clustering tolerance
agg_b_init, agg_te_init, agg_d_init, sizes_init, counts_init = (
    aggregate_envelope_points(b_init_f, te_init_f, d_init_f)
)
agg_b_opt, agg_te_opt, agg_d_opt, sizes_opt, counts_opt = (
    aggregate_envelope_points(b_opt_f, te_opt_f, d_opt_f)
)

# -------------------------------------------------------------
# 3. Dynamic Normalization & Shared Axis Limits
# -------------------------------------------------------------
d_all = np.concatenate([d_init_f, d_opt_f])
vmin_delta = float(np.nanmin(d_all))
vmax_delta = float(np.nanmax(d_all))
if np.isclose(vmin_delta, vmax_delta):
    vmax_delta = vmin_delta + 1.0
delta_norm = mcolors.Normalize(vmin=vmin_delta, vmax=vmax_delta)

b_all = np.concatenate([b_init_f, b_opt_f])
min_b = float(np.nanmin(b_all))
max_b = float(np.nanmax(b_all))
if np.isclose(min_b, max_b):
    max_b = min_b + 100.0

b_norm = mcolors.Normalize(vmin=max(0.0, min_b), vmax=max_b)

b_margin = max(50.0, 0.08 * (max_b - min_b))
xlim_shared = (-b_margin, max_b + b_margin * 1.5)

te_all = np.concatenate([te_init_f, te_opt_f])
te_min, te_max = float(np.nanmin(te_all)), float(np.nanmax(te_all))
te_margin = 0.12 * (te_max - te_min) if not np.isclose(te_min, te_max) else 8.0
ylim_shared = (max(0.0, te_min - te_margin), te_max + te_margin)

# -------------------------------------------------------------
# 4. Render Figure (2x2)
# -------------------------------------------------------------
fig = plt.figure(figsize=(16, 12))

u = np.linspace(0, 2 * np.pi, 25)
v = np.linspace(0, np.pi, 25)
sphere_x = np.outer(np.cos(u), np.sin(v))
sphere_y = np.outer(np.sin(u), np.sin(v))
sphere_z = np.outer(np.ones(np.size(u)), np.cos(v))

# --- Panel 1: Initial Scheme Envelope ---
ax1 = fig.add_subplot(2, 2, 1)
sc1 = plot_concentric_envelope_scatter(
    ax1, agg_b_init, agg_te_init, agg_d_init, sizes_init, counts_init, delta_norm, is_opt=False
)
ax1.axvline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.7, zorder=1)
ax1.set_xlim(xlim_shared)
ax1.set_ylim(ylim_shared)
ax1.set_xlabel(r"$b$-value ($\mathrm{s/mm}^2$)", fontsize=11)
ax1.set_ylabel(r"$\mathrm{TE}$ ($\mathrm{ms}$)", fontsize=11)
ax1.set_title("Initial Scheme: Envelope Space", fontsize=12, fontweight="bold")
ax1.grid(True, linestyle="--", alpha=0.5, zorder=1)

cbar1 = fig.colorbar(sc1, ax=ax1, pad=0.02)
cbar1.set_label(r"Effective Diffusion Time $\Delta - \delta/3$ ($\mathrm{ms}$)", fontsize=10)

# --- Panel 2: Initial Scheme Directions ---
ax2 = fig.add_subplot(2, 2, 2, projection="3d")
ax2.plot_wireframe(sphere_x, sphere_y, sphere_z, color="gray", alpha=0.2)
sc2 = ax2.scatter(
    udirs_init_f[:, 0],
    udirs_init_f[:, 1],
    udirs_init_f[:, 2],
    c=b_init_f,
    cmap="plasma",
    norm=b_norm,
    s=65,
    edgecolors="k",
)
cbar2 = fig.colorbar(sc2, ax=ax2, pad=0.08, shrink=0.75)
cbar2.set_label(r"$b$-value ($\mathrm{s/mm}^2$)", fontsize=10)
ax2.set_title("Initial Scheme: Directions ($\hat{\mathbf{g}}$)", fontsize=12, fontweight="bold")
ax2.set_xlim([-1.1, 1.1])
ax2.set_ylim([-1.1, 1.1])
ax2.set_zlim([-1.1, 1.1])
ax2.set_xlabel("gx")
ax2.set_ylabel("gy")
ax2.set_zlabel("gz")

# --- Panel 3: Optimized Scheme Envelope ---
ax3 = fig.add_subplot(2, 2, 3, sharex=ax1, sharey=ax1)
sc3 = plot_concentric_envelope_scatter(
    ax3, agg_b_opt, agg_te_opt, agg_d_opt, sizes_opt, counts_opt, delta_norm, is_opt=True
)
ax3.axvline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.7, zorder=1)
ax3.set_xlim(xlim_shared)
ax3.set_ylim(ylim_shared)
ax3.set_xlabel(r"$b$-value ($\mathrm{s/mm}^2$)", fontsize=11)
ax3.set_ylabel(r"$\mathrm{TE}$ ($\mathrm{ms}$)", fontsize=11)
ax3.set_title("Optimized Scheme: Envelope Space", fontsize=12, fontweight="bold")
ax3.grid(True, linestyle="--", alpha=0.5, zorder=1)

cbar3 = fig.colorbar(sc3, ax=ax3, pad=0.02)
cbar3.set_label(r"Effective Diffusion Time $\Delta - \delta/3$ ($\mathrm{ms}$)", fontsize=10)

# --- Panel 4: Optimized Scheme Directions ---
ax4 = fig.add_subplot(2, 2, 4, projection="3d")
ax4.plot_wireframe(sphere_x, sphere_y, sphere_z, color="gray", alpha=0.2)
sc4 = ax4.scatter(
    udirs_opt_f[:, 0],
    udirs_opt_f[:, 1],
    udirs_opt_f[:, 2],
    c=b_opt_f,
    cmap="plasma",
    norm=b_norm,
    s=65,
    edgecolors="k",
)
cbar4 = fig.colorbar(sc4, ax=ax4, pad=0.08, shrink=0.75)
cbar4.set_label(r"$b$-value ($\mathrm{s/mm}^2$)", fontsize=10)
ax4.set_title("Optimized Scheme: Directions ($\hat{\mathbf{g}}$)", fontsize=12, fontweight="bold")
ax4.set_xlim([-1.1, 1.1])
ax4.set_ylim([-1.1, 1.1])
ax4.set_zlim([-1.1, 1.1])
ax4.set_xlabel("gx")
ax4.set_ylabel("gy")
ax4.set_zlabel("gz")

plt.tight_layout()
plt.savefig(plotdir / "scheme_comparison_envelope.png", dpi=300)
plt.show()

# -------------------------------------------------------------
# 5. Summary Breakdown
# -------------------------------------------------------------
print("\n=== Optimized Shells (Excluding Fixed Baseline b0) ===")
unique_coords, counts = np.unique(
    np.column_stack([np.round(b_opt_f, -1), np.round(te_opt_f, 1), np.round(d_opt_f, 1)]),
    axis=0,
    return_counts=True,
)
for (b_val, te_val, d_val), count in zip(unique_coords, counts):
    print(
        f"b ≈ {int(b_val):4d} s/mm² | Directions: {count:2d} | "
        f"TE: {te_val:.1f} ms | "
        f"Δ_eff: {d_val:.1f} ms"
    )

numpy.core.multiarray /nfs/rtfs03/storage/home/jgroot34/miniforge3/envs/MICROtool/lib/python3.11/site-packages/numpy/core/multiarray.py
numpy._core.multiarray /nfs/rtfs03/storage/home/jgroot34/miniforge3/envs/MICROtool/lib/python3.11/site-packages/numpy/core/multiarray.py


NameError: name 'make_expirement_directories' is not defined